# Lick Event Labeling Tool

**Kernel: cliqr-network-training**

Label capacitive traces for ML training. One ~1-minute chunk at a time, random order.  
First and last 10 chunks (relative to the recorded start/stop times) are excluded.

| Action | Control |
|--------|---------|
| Add lick label | Left-click on trace |
| Remove nearest label | Right-click on trace |
| Confirm & advance | **✓ Done →** |
| Skip without labeling | **Skip** |
| Go back | **← Back** |

> Labels auto-snap left to the earliest point within ±5% of the clicked capacitance value.  
> Zoom/pan with the toolbar freely — labels are blocked while any toolbar mode is active.

In [1]:
%matplotlib widget
import re
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
import json
import glob
import random
from pathlib import Path
from datetime import datetime
import torch


In [2]:
CHUNK_DURATION_S     = 60    # seconds per chunk
SKIP_FIRST_N         = 5    # discard N chunks from recording start
SKIP_LAST_N          = 5    # discard N chunks from recording stop
SNAP_S               = 0.075  # minimum distance between labels / right-click snap radius (seconds)
MIN_DYNAMIC_RANGE_PF = 5    # discard chunks with max-min capacitance below this (pF)
DATA_DIR             = 'Lickometry Data'
OUTPUT_DIR           = 'Training Data'

In [3]:
h5_files = sorted(glob.glob(f'{DATA_DIR}/**/*.h5', recursive=True))
if not h5_files:
    print(f'No .h5 files found under {DATA_DIR}/')

file_select = widgets.Dropdown(
    options=h5_files,
    description='File:',
    layout=widgets.Layout(width='600px'),
)
load_btn    = widgets.Button(description='Load File', button_style='primary', icon='folder-open')
file_status = widgets.Label(value='Select a file and click Load.')
traces      = {}


def _get_recording_window(grp):
    """Return (rec_start, rec_stop) absolute timestamps for an HDF5 group.

    Mirrors the start_time/stop_time resolution logic in data_analysis.filter_data:
    finds the highest-numbered start_timeN key and its matching stop_timeN.
    Falls back to time_data[0] / time_data[-1] when keys are absent.
    """
    time_data = grp['time_data'][()]
    pattern   = re.compile(r'^start_time(\d+)?$')
    matches   = {}
    for k in grp.keys():
        m = pattern.match(k)
        if m:
            num = int(m.group(1)) if m.group(1) else -1
            matches[num] = k

    if matches:
        last_key  = matches[max(matches.keys())]
        rec_start = float(grp[last_key][()])
        stop_key  = 'stop' + last_key[5:]
        rec_stop  = float(grp[stop_key][()]) if stop_key in grp else float(time_data[-1])
    else:
        rec_start = float(time_data[0])
        rec_stop  = float(time_data[-1])

    return rec_start, rec_stop


def on_load(b):
    global traces
    traces = {}
    path = file_select.value
    try:
        with h5py.File(path, 'r') as f:
            def _visit(name, obj):
                if not (isinstance(obj, h5py.Group)
                        and 'cap_data' in obj
                        and 'time_data' in obj):
                    return
                rec_start, rec_stop = _get_recording_window(obj)
                entry = {
                    'time':            obj['time_data'][()],
                    'cap':             obj['cap_data'][()],
                    'recording_start': rec_start,
                    'recording_stop':  rec_stop,
                }
                if 'lick_times' in obj:
                    entry['algo_lick_times'] = obj['lick_times'][()]
                traces[name] = entry
            f.visititems(_visit)
        file_status.value = f'Loaded {len(traces)} traces  |  {Path(path).name}'
    except Exception as e:
        file_status.value = f'Error: {e}'


load_btn.on_click(on_load)
display(widgets.HBox([file_select, load_btn]))
display(file_status)


Label(value='Select a file and click Load.')

In [4]:
def _segment(trace_key, td, chunk_s, skip_first, skip_last):
    """Split a trace into fixed-length chunks anchored to the recorded
    start/stop window, then discard the first and last N chunks and any
    chunk whose dynamic range is below MIN_DYNAMIC_RANGE_PF."""
    time      = td['time']
    cap       = td['cap']
    rec_start = td['recording_start']
    rec_stop  = td['recording_stop']

    n_total = int((rec_stop - rec_start) // chunk_s)
    chunks  = []
    for i in range(n_total):
        ts        = rec_start + i * chunk_s
        te        = ts + chunk_s
        mask      = (time >= ts) & (time < te)
        if mask.sum() < 5:
            continue
        chunk_cap = cap[mask]
        if chunk_cap.max() - chunk_cap.min() < MIN_DYNAMIC_RANGE_PF:
            continue
        c = {
            'trace_key': trace_key,
            'chunk_idx': i,
            'time':      time[mask] - ts,   # 0-relative within chunk
            'cap':       chunk_cap,
        }
        if 'algo_lick_times' in td:
            alt = td['algo_lick_times']
            c['algo_lick_times'] = alt[(alt >= ts) & (alt < te)] - ts
        chunks.append(c)

    n = len(chunks)
    return chunks[skip_first : n - skip_last] if n > skip_first + skip_last else []


if not traces:
    print('Run the File Selector cell first and load a file.')
else:
    chunk_list  = []
    n_discarded = 0
    for tk, td in traces.items():
        # count raw chunks (before edge discard) to compute how many were dropped by dynamic range
        time      = td['time']
        rec_start = td['recording_start']
        rec_stop  = td['recording_stop']
        n_raw     = int((rec_stop - rec_start) // CHUNK_DURATION_S)

        segs = _segment(tk, td, CHUNK_DURATION_S, SKIP_FIRST_N, SKIP_LAST_N)
        chunk_list.extend(segs)

        n_after_edge = max(n_raw - SKIP_FIRST_N - SKIP_LAST_N, 0)
        n_dropped    = n_after_edge - len(segs)
        n_discarded += n_dropped
        dur = rec_stop - rec_start
        print(f'  {tk}: {dur/60:.1f} min  →  {len(segs)} chunks  ({n_dropped} low-range dropped)')

    print(f'\nTotal usable chunks: {len(chunk_list)}  ({n_discarded} discarded for low dynamic range)')
    chunk_lookup = {(c['trace_key'], c['chunk_idx']): c for c in chunk_list}

    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    PROGRESS_FILE = Path(OUTPUT_DIR) / (Path(file_select.value).stem + '.progress.json')

    labels      = {}   # (trace_key, chunk_idx) -> list[float] lick times
    labeled_set = set()
    skipped_set = set()
    _start_pos  = 0

    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            prog = json.load(f)
        shuffled_order = [(item[0], int(item[1])) for item in prog['shuffled_order']]
        # drop any entries that were removed by the dynamic range filter
        shuffled_order = [key for key in shuffled_order if tuple(key) in chunk_lookup
                          or (key[0], key[1]) in chunk_lookup]
        shuffled_order = [(tk, ci) for tk, ci in shuffled_order if (tk, ci) in chunk_lookup]
        for k, v in prog.get('labels', {}).items():
            parts = k.split('|||', 1)
            labels[(parts[0], int(parts[1]))] = v
        labeled_set = {(a, int(b)) for a, b in prog.get('labeled_set', [])}
        skipped_set = {(a, int(b)) for a, b in prog.get('skipped_set', [])}
        _start_pos  = prog.get('current_pos', 0)
        print(f'Resumed: {len(labeled_set)} done, {len(skipped_set)} skipped, pos={_start_pos}')
    else:
        shuffled_order = [(c['trace_key'], c['chunk_idx']) for c in chunk_list]
        random.shuffle(shuffled_order)
        print('Fresh session — order shuffled.')


  board_FT232H0/sensor_1: 121.1 min  →  34 chunks  (77 low-range dropped)
  board_FT232H0/sensor_2: 120.6 min  →  22 chunks  (88 low-range dropped)
  board_FT232H0/sensor_3: 120.6 min  →  9 chunks  (101 low-range dropped)
  board_FT232H0/sensor_7: 120.1 min  →  13 chunks  (97 low-range dropped)
  board_FT232H0/sensor_8: 119.8 min  →  5 chunks  (104 low-range dropped)
  board_FT232H0/sensor_9: 119.9 min  →  40 chunks  (69 low-range dropped)
  board_FT232H1/sensor_10: 120.0 min  →  11 chunks  (99 low-range dropped)
  board_FT232H1/sensor_11: 120.1 min  →  3 chunks  (107 low-range dropped)
  board_FT232H1/sensor_12: 120.2 min  →  12 chunks  (98 low-range dropped)
  board_FT232H1/sensor_4: 120.6 min  →  17 chunks  (93 low-range dropped)
  board_FT232H1/sensor_5: 120.3 min  →  49 chunks  (61 low-range dropped)
  board_FT232H1/sensor_6: 120.4 min  →  25 chunks  (85 low-range dropped)
  board_FT232H2/sensor_13: 120.1 min  →  15 chunks  (95 low-range dropped)
  board_FT232H2/sensor_14: 120.1 m

In [ ]:
from threading import Timer as _Timer

plt.close('all')
with plt.ioff():
    fig, ax = plt.subplots(figsize=(14, 4))
fig.tight_layout(pad=1.5)

_state = {'pos': _start_pos, 'vlines': [], 'algo_vlines': []}

# Widgets
progress_lbl = widgets.HTML(value='')
count_lbl    = widgets.Label(value='', layout=widgets.Layout(width='90px'))
status_lbl   = widgets.Label(value='')
show_algo    = widgets.Checkbox(value=True, description='Show algo', indent=False,
                                layout=widgets.Layout(width='120px'))

back_btn = widgets.Button(description='← Back',   button_style='',        layout=widgets.Layout(width='100px'))
skip_btn = widgets.Button(description='Skip',      button_style='warning', layout=widgets.Layout(width='90px'))
done_btn = widgets.Button(description='✓ Done →', button_style='success', layout=widgets.Layout(width='110px'))
save_btn = widgets.Button(description='Save',      button_style='info',    icon='save',
                          layout=widgets.Layout(width='90px'))


def _cur_key():
    pos = _state['pos']
    return shuffled_order[pos] if pos < len(shuffled_order) else None


def _update_progress():
    pos   = _state['pos']
    total = len(shuffled_order)
    progress_lbl.value = (
        f'<b>Chunk {min(pos + 1, total)} / {total}</b>'
        f' &nbsp;|&nbsp; Done: {len(labeled_set)}'
        f' &nbsp;|&nbsp; Skipped: {len(skipped_set)}'
    )


def _update_count():
    key = _cur_key()
    count_lbl.value = f'Licks: {len(labels.get(key, []))}'


def _redraw_licks():
    key = _cur_key()
    for vl in _state['vlines']:
        vl.remove()
    _state['vlines'] = []
    for lt in labels.get(key, []):
        _state['vlines'].append(ax.axvline(lt, color='red', alpha=0.8, linewidth=1.2))
    _update_count()
    fig.canvas.draw()


def _snap_to_peak(chunk, x, threshold_frac=0.01, max_lookback_s=1.0):
    """Walk left from click while cap doesn't rise more than 5% above the clicked value,
    capped at max_lookback_s before the click."""
    time = chunk['time']
    cap  = chunk['cap']
    n    = len(time)

    click_idx = max(int(np.searchsorted(time, x, side='right')) - 1, 0)
    limit_idx = int(np.searchsorted(time, max(float(time[0]), float(time[click_idx]) - max_lookback_s)))
    v_hi      = float(cap[click_idx]) * (1.0 + threshold_frac)

    result_idx = click_idx
    for i in range(click_idx - 1, limit_idx - 1, -1):
        if float(cap[i]) <= v_hi:
            result_idx = i
        else:
            break

    return float(time[result_idx])


def _draw_chunk():
    key = _cur_key()
    if key is None:
        ax.cla()
        ax.text(0.5, 0.5, 'All chunks visited!', transform=ax.transAxes,
                ha='center', va='center', fontsize=16)
        fig.canvas.draw()
        _update_progress()
        return

    tk, ci = key
    if key not in chunk_lookup:
        status_lbl.value = f'Chunk {key} missing from data — skipping'
        _advance(to_labeled=False)
        return

    chunk = chunk_lookup[key]
    ax.cla()
    _state['vlines']      = []
    _state['algo_vlines'] = []

    ax.plot(chunk['time'], chunk['cap'], color='steelblue', linewidth=0.6)
    ax.set_xlabel('Time within chunk (s)')
    ax.set_ylabel('Capacitance (pF)')
    tag = 'Done ✓' if key in labeled_set else ('Skipped' if key in skipped_set else 'Unvisited')
    ax.set_title(f'{tk}  |  chunk {ci}  |  {tag}', fontsize=9)

    if show_algo.value and 'algo_lick_times' in chunk:
        for lt in chunk['algo_lick_times']:
            vl = ax.axvline(lt, color='orange', alpha=0.5, linewidth=1.0, linestyle='--')
            _state['algo_vlines'].append(vl)

    for lt in labels.get(key, []):
        _state['vlines'].append(ax.axvline(lt, color='red', alpha=0.8, linewidth=1.2))

    legend = [mpatches.Patch(color='red', label='manual label')]
    if show_algo.value and 'algo_lick_times' in chunk:
        legend.append(mpatches.Patch(color='orange', label='algo detection'))
    ax.legend(handles=legend, loc='upper right', fontsize=8)

    _update_count()
    _update_progress()
    fig.canvas.draw()


def _save_progress():
    data = {
        'shuffled_order': [[tk, ci] for tk, ci in shuffled_order],
        'labels':         {f'{tk}|||{ci}': v for (tk, ci), v in labels.items()},
        'labeled_set':    [[tk, ci] for tk, ci in labeled_set],
        'skipped_set':    [[tk, ci] for tk, ci in skipped_set],
        'current_pos':    _state['pos'],
    }
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(data, f, indent=2)


def _advance(to_labeled):
    key = _cur_key()
    if key:
        (labeled_set if to_labeled else skipped_set).add(key)
    _state['pos'] = min(_state['pos'] + 1, len(shuffled_order))
    _save_progress()
    _draw_chunk()
    status_lbl.value = ''


def on_click(event):
    if event.inaxes is not ax or event.xdata is None:
        return
    key = _cur_key()
    if not key or key not in chunk_lookup:
        return
    x = event.xdata

    if event.button == 1:
        try:
            toolbar_busy = fig.canvas.toolbar.mode != ''
        except AttributeError:
            toolbar_busy = False
        if toolbar_busy:
            return
        chunk  = chunk_lookup[key]
        x_peak = _snap_to_peak(chunk, x)
        # Block if click OR snapped position is near an existing label
        if any(abs(x - lt) < SNAP_S or abs(x_peak - lt) < SNAP_S for lt in labels.get(key, [])):
            status_lbl.value = 'Too close to existing label — right-click to remove'
            return
        labels.setdefault(key, []).append(x_peak)
        labels[key].sort()
        snapped = abs(x_peak - x) > 0.005
        status_lbl.value = (
            f'Added lick at {x_peak:.3f} s'
            + (f'  (snapped from {x:.3f} s)' if snapped else '')
        )

    elif event.button == 3:
        lts = labels.get(key, [])
        if not lts:
            return
        dists = [abs(x - lt) for lt in lts]
        idx   = int(np.argmin(dists))
        if dists[idx] < SNAP_S:
            removed = lts.pop(idx)
            status_lbl.value = f'Removed lick at {removed:.3f} s'
        else:
            status_lbl.value = f'No label within {SNAP_S} s'

    _redraw_licks()


fig.canvas.mpl_connect('button_press_event', on_click)
done_btn.on_click(lambda b: _advance(to_labeled=True))
skip_btn.on_click(lambda b: _advance(to_labeled=False))


def on_back(b):
    _state['pos'] = max(_state['pos'] - 1, 0)
    _draw_chunk()
    status_lbl.value = ''


def on_save(b):
    _save_progress()
    status_lbl.value = f'Progress saved → {PROGRESS_FILE.name}'


back_btn.on_click(on_back)
save_btn.on_click(on_save)
show_algo.observe(lambda change: _draw_chunk(), names='value')

toolbar_row = widgets.HBox([back_btn, skip_btn, done_btn, count_lbl, save_btn, show_algo])
display(progress_lbl)
display(fig.canvas)
display(toolbar_row)
display(status_lbl)

_Timer(0.5, _draw_chunk).start()

In [5]:
if not labeled_set:
    print('No chunks labeled yet.')
else:
    samples = []
    for key in sorted(labeled_set):
        tk, ci = key
        if key not in chunk_lookup:
            print(f'Warning: {key} not in chunk_lookup, skipping')
            continue
        chunk      = chunk_lookup[key]
        lick_times = sorted(labels.get(key, []))
        samples.append({
            'cap':         torch.tensor(chunk['cap'],  dtype=torch.float32),
            'time':        torch.tensor(chunk['time'], dtype=torch.float32),
            'lick_times':  torch.tensor(lick_times,    dtype=torch.float32),
            'source_file': str(file_select.value),
            'trace_key':   tk,
            'chunk_idx':   ci,
        })

    output_path = Path(OUTPUT_DIR) / (Path(file_select.value).stem + '.pt')
    torch.save({
        'samples': samples,
        'metadata': {
            'source_file':      str(file_select.value),
            'chunk_duration_s': CHUNK_DURATION_S,
            'skip_first_n':     SKIP_FIRST_N,
            'skip_last_n':      SKIP_LAST_N,
            'n_samples':        len(samples),
            'n_with_licks':     sum(1 for s in samples if len(s['lick_times']) > 0),
            'n_no_licks':       sum(1 for s in samples if len(s['lick_times']) == 0),
            'created':          datetime.now().isoformat(),
        },
    }, output_path)

    n_pos = sum(1 for s in samples if len(s['lick_times']) > 0)
    n_neg = len(samples) - n_pos
    print(f'Saved {len(samples)} labeled chunks → {output_path}')
    print(f'  Chunks with licks:    {n_pos}')
    print(f'  Chunks without licks: {n_neg}')


Saved 108 labeled chunks → Training Data/raw_data_2025-09-01_10-57-58.pt
  Chunks with licks:    32
  Chunks without licks: 76


In [ ]:
output_path = Path(OUTPUT_DIR) / (Path(file_select.value).stem + '.pt')
if output_path.exists():
    data    = torch.load(output_path, weights_only=False)
    meta    = data['metadata']
    samples = data['samples']
    print(f"Source:          {meta['source_file']}")
    print(f"Chunk duration:  {meta['chunk_duration_s']} s")
    print(f"Total samples:   {meta['n_samples']}")
    print(f"  With licks:    {meta['n_with_licks']}")
    print(f"  Without licks: {meta['n_no_licks']}")
    if samples:
        s = samples[0]
        print(f"\nSample [0]:")
        print(f"  cap shape:   {s['cap'].shape}  dtype: {s['cap'].dtype}")
        print(f"  time shape:  {s['time'].shape}")
        print(f"  lick_times:  {s['lick_times']}")
        print(f"  trace_key:   {s['trace_key']}  chunk: {s['chunk_idx']}")
